# Kết quả `collect.py`

Đọc `<paths.raw>` (mặc định `data/raw/`): `classified.jsonl` (quyết định LẤY/BỎ từng file), `raw_index.jsonl` (file đã lấy), `files/<id>_transcript.txt` và `_turnevent.txt`, cùng `manifests/*.jsonl` (segment). Chạy lại notebook bất cứ lúc nào, kể cả khi `collect.py` đang chạy.

Muốn kiểm tra nhãn LLM gán có đúng không thì dùng `inspect_answers.ipynb`.

In [ ]:
import collections, glob, json, os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "tools"))
import conf
from ask_models import load_jsonl
from build_manifest import load_tr, load_ev

C = conf.load([])        # vd. conf.load(["collect=local"]) hoặc conf.load(["paths.raw=data/raw_smoke"])
RAW, MANIF = conf.path(C, "raw"), conf.path(C, "manifests")
FILES = os.path.join(RAW, "files")

def jl(p):
    return load_jsonl(p) if os.path.exists(p) else []

cls = list({r["id"]: r for r in jl(os.path.join(RAW, "classified.jsonl"))}.values())   # dòng mới nhất mỗi file
for r in cls:
    r.setdefault("decision", "LẤY" if r.get("used") else "BỎ"); r.setdefault("reason", "")
idx = jl(os.path.join(RAW, "raw_index.jsonl"))
segs = [r for p in sorted(glob.glob(os.path.join(MANIF, "*.jsonl"))) for r in load_jsonl(p)]
print(f"{RAW}\n{len(cls)} file đã quyết định · {len(idx)} file LẤY · {len(segs)} segment")

## Quota theo bucket

In [ ]:
import pandas as pd
have = collections.Counter()
for r in idx:
    for b in r["buckets"]:
        have[b] += r["speech_min"]
Q = pd.DataFrame([{"bucket": b["name"], "mô tả": b["desc"], "có (phút)": round(have[b["name"]], 1), "mục tiêu": b["target_minutes"],
                   "đạt %": round(100 * have[b["name"]] / b["target_minutes"])} for b in C["collect"]["buckets"]])
Q.style.bar(subset=["đạt %"], vmin=0, vmax=100, color="#9fd3a0")

## Quyết định LẤY / BỎ

In [ ]:
import re
D = pd.DataFrame([{"id": r["id"], "file": os.path.basename(r["src"]), "quyết định": r["decision"], "lý do": r["reason"],
                   "lý do (nhóm)": re.sub(r"\(.*?\)|[\d.]+", "", r["reason"]).strip(), "phút": r.get("speech_min"),
                   "lỗi parse": r.get("parse_error"), **r["labels"], "src": r["src"]} for r in cls])
display(D.groupby(["quyết định", "lý do (nhóm)"]).size().rename("số file").reset_index().sort_values("số file", ascending=False)) if len(D) else print("Chưa có file nào.")

## Phân bố nhãn LLM gán (mọi file đã phân loại)

In [ ]:
import matplotlib.pyplot as plt
cols = [c for c in ["genre", "channel", "noise_type", "distance", "accent", "speakers"] if c in D]
if cols:
    fig, axes = plt.subplots(1, len(cols), figsize=(3.2 * len(cols), 3.2), squeeze=False)
    for ax, c in zip(axes[0], cols):
        t = D.groupby([c, "quyết định"]).size().unstack(fill_value=0)
        t.plot.barh(ax=ax, stacked=True, legend=ax is axes[0][0], color={"LẤY": "#4c9a5f", "BỎ": "#d9a441"})
        ax.set_title(c); ax.set_ylabel(""); ax.set_xlabel("số file")
        ax.xaxis.get_major_locator().set_params(integer=True)
    plt.tight_layout()

## Segment đã cắt

In [ ]:
S = pd.DataFrame([{"event_id": s["event_id"], "file": s["conversation_id"], "event": s["event_auto"],
                   "bucket": ", ".join(s.get("bucket") or []), "gap_sau (s)": s.get("gap_after_sec_auto"),
                   "chồng lời": s.get("overlap_auto"), "lời người dùng": s.get("user_text", "")[:80]} for s in segs])
if len(S):
    display(pd.crosstab(S["bucket"], S["event"], margins=True, margins_name="tổng"))
    fig, ax = plt.subplots(figsize=(6, 2.6))
    for e, g in S.groupby("event"):
        ax.hist(g["gap_sau (s)"].dropna().clip(upper=3), bins=30, alpha=0.6, label=e)
    ax.set_xlabel("khoảng lặng sau sự kiện (s, cắt ở 3)"); ax.legend(); plt.tight_layout()
else:
    print("Chưa có segment.")

## Xem từng file

Dòng thời gian: mỗi hàng một người nói (từ diarization), vạch dọc là sự kiện gán tự động. Bên dưới là transcript và các segment đã cắt, kèm clip để nghe.

In [ ]:
from IPython.display import Audio, HTML, display
EV_COLOR = {"EoT": "#2b7bba", "MoT": "#e08a1e", "BC": "#3a9a4a", "INT": "#c0392b", "FI": "#e08a1e"}

def timeline(fid, t0=0, t1=None):
    tp, ep = os.path.join(FILES, fid + "_transcript.txt"), os.path.join(FILES, fid + "_turnevent.txt")
    if not os.path.exists(tp):
        print("Không có transcript (file bị BỎ thì wav và transcript đã xoá, chỉ còn nhãn)."); return [], []
    tr, ev = load_tr(tp), (load_ev(ep) if os.path.exists(ep) else [])
    t1 = t1 or max(t["e"] for t in tr)
    spk = sorted({t["spk"] for t in tr})
    fig, ax = plt.subplots(figsize=(14, 0.5 + 0.45 * len(spk)))
    for t in tr:
        if t["e"] >= t0 and t["s"] <= t1:
            ax.barh(spk.index(t["spk"]), t["e"] - t["s"], left=t["s"], height=0.6, color="#9db7d5")
    for s, e, c in ev:
        if t0 <= s <= t1:
            ax.axvline(s, color=EV_COLOR.get(c, "k"), lw=1.2, alpha=0.9)
    ax.set_yticks(range(len(spk)), spk); ax.set_xlim(t0, t1); ax.set_xlabel("giây")
    ax.legend(handles=[plt.Line2D([], [], color=v, label=k) for k, v in EV_COLOR.items() if k != "FI"], loc="lower right", bbox_to_anchor=(1, 1), ncol=4, fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()
    return tr, ev

def show_file(fid, t0=0, t1=120, n_segments=8):
    r = next((x for x in cls if x["id"] == fid), {})
    display(HTML(f"<h3>{os.path.basename(r.get('src', fid))} · <b>{r.get('decision', '?')}</b> · {r.get('reason', '')}</h3>"
                 f"<p>Nhãn LLM: {' · '.join(f'{k}={v}' for k, v in (r.get('labels') or {}).items())}<br><small>{r.get('src', '')}</small></p>"))
    wav = os.path.join(FILES, fid + ".wav")
    if os.path.exists(wav):
        display(Audio(wav))
    tr, ev = timeline(fid, t0, t1)
    if tr:
        rows = [(f"{t['s']:7.2f}–{t['e']:7.2f}", t["spk"], t["text"]) for t in tr if t["e"] >= t0 and t["s"] <= t1]
        display(pd.DataFrame(rows, columns=["thời gian", "người nói", "lời"]).style.set_properties(subset=["lời"], **{"text-align": "left"}))
    mine = [s for s in segs if s["conversation_id"] == fid]
    if mine:
        display(HTML(f"<h4>{len(mine)} segment (hiện {min(n_segments, len(mine))})</h4>"))
        for s in mine[:n_segments]:
            display(HTML(f"<b>{s['event_id']}</b> · <span style='color:{EV_COLOR.get(s['event_raw'], '#000')}'><b>{s['event_auto']}</b></span>"
                         f" · times={s.get('times')} · next={s.get('next_auto')} · gap={s.get('gap_after_sec_auto')}s<br><i>{s.get('user_text', '')}</i>"))
            p = os.path.join(ROOT, s["audio"]) if s.get("audio") else None
            if p and os.path.exists(p):
                display(Audio(p))

FILE_ID = None          # đặt id cụ thể; None = file LẤY mới nhất
taken = [r["id"] for r in cls if r["decision"] == "LẤY"]
fid = FILE_ID or (taken[-1] if taken else (cls[-1]["id"] if cls else None))
show_file(fid) if fid else print("Chưa có file nào.")

### Chọn file tương tác (cần `ipywidgets`)

In [ ]:
try:
    import ipywidgets as w
    dec = w.Dropdown(options=["LẤY", "BỎ", "tất cả"], description="quyết định")
    sel = w.Dropdown(description="file", layout=w.Layout(width="70%"))
    rng = w.FloatRangeSlider(value=(0, 120), min=0, max=600, step=5, description="đoạn (s)", layout=w.Layout(width="70%"))
    out = w.Output()
    def refresh(*_):
        sel.options = [(f"{os.path.basename(r['src'])} · {r['reason']}", r["id"]) for r in cls if dec.value == "tất cả" or r["decision"] == dec.value] or [("(trống)", None)]
    def render(*_):
        out.clear_output()
        with out:
            if sel.value:
                show_file(sel.value, *rng.value)
    dec.observe(refresh, "value"); sel.observe(render, "value"); rng.observe(render, "value")
    refresh(); display(dec, sel, rng, out)
except ImportError:
    print("Không có ipywidgets: đặt FILE_ID ở cell trên.")